# TechMentor Inference

Test the fine-tuned LoRA adapter on interview mentor tasks.

**Requirements:** CUDA GPU, Hugging Face access to `meta-llama/Llama-3.2-3B-Instruct`, trained adapter at `training/outputs/final_adapter/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from training.config import FINAL_ADAPTER_DIR, MODEL_ID
from training.prompts import TEST_PROMPTS, format_inference_prompt

In [ ]:
ADAPTER_PATH = ROOT / "training" / "outputs" / "final_adapter"
MAX_NEW_TOKENS = 256

assert torch.cuda.is_available(), "CUDA GPU required for 4-bit inference"
assert ADAPTER_PATH.exists(), f"Train first. Adapter not found: {ADAPTER_PATH}"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model.eval()

In [ ]:
def generate(instruction: str, user_input: str) -> str:
    prompt = format_inference_prompt(tokenizer, instruction, user_input)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## Test prompts

In [ ]:
for item in TEST_PROMPTS:
    print("=" * 80)
    print(f"[{item['category']}]")
    print(f"Instruction: {item['instruction']}")
    print(f"Input: {item['input']}")
    print("-" * 80)
    print(generate(item["instruction"], item["input"]))
    print()

## Custom prompt

In [ ]:
instruction = "Answer the technical interview question."
user_input = "What is the CAP theorem?"
print(generate(instruction, user_input))